In [1]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import itertools
import sys
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn import metrics as metrics
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv("UNSW_NB15_training-set.csv" ) 


test = pd.read_csv("UNSW_NB15_testing-set.csv" )


In [3]:
y = train["attack_cat"].values
from collections import Counter
Counter(y)

Counter({'Normal': 37000,
         'Reconnaissance': 3496,
         'Backdoor': 583,
         'DoS': 4089,
         'Exploits': 11132,
         'Analysis': 677,
         'Fuzzers': 6062,
         'Worms': 44,
         'Shellcode': 378,
         'Generic': 18871})

In [4]:
y1 = test["attack_cat"].values
from collections import Counter
Counter(y1)

Counter({'Normal': 56000,
         'Backdoor': 1746,
         'Analysis': 2000,
         'Fuzzers': 18184,
         'Shellcode': 1133,
         'Reconnaissance': 10491,
         'Exploits': 33393,
         'DoS': 12264,
         'Worms': 130,
         'Generic': 40000})

In [5]:
from sklearn.preprocessing import LabelEncoder
encodings = dict()
for c in train.columns:
    
    if train[c].dtype == "object":
        encodings[c] = LabelEncoder()
        encodings[c]
        train[c] = encodings[c].fit_transform(train[c])

In [6]:
y = train.pop("attack_cat").values
X = train.values
X_ICA = X
y_ICA = y

In [7]:
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X)

In [8]:
#ICA
from sklearn.decomposition import FastICA
np.random.seed(0)
ica = FastICA(n_components=10)
S = ica.fit_transform(X_ICA)

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
#KNN
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=5,weights='uniform')
neigh.fit(X_train, y_train)

KNeighborsClassifier()

In [29]:
y_pred = neigh.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7427, 5: 3633, 3: 2463, 4: 1282, 2: 790, 7: 648, 0: 150, 1: 41, 8: 33})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [30]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   9    2   22   73   24    0    1    0    0    0]
 [   5    0    7   75   27    1    0    2    0    0]
 [   9   13  312  374   42    5    1   23    7    0]
 [  82   12  331 1555  188    8    1   91    7    0]
 [  45    8   47  169  887    7    7   39    3    0]
 [   0    1   19   69   14 3609    1   10    0    0]
 [   0    0    0    0    1    1 7416    0    0    0]
 [   0    5   52  131   80    0    0  453    2    0]
 [   0    0    0   11   18    2    0   30   14    0]
 [   0    0    0    6    1    0    0    0    0    0]]
Accuracy Score : 0.8656707354102143
Report : 
              precision    recall  f1-score   support

           0       0.06      0.07      0.06       131
           1       0.00      0.00      0.00       117
           2       0.39      0.40      0.40       786
           3       0.63      0.68      0.66      2275
           4       0.69      0.73      0.71      1212
           5       0.99      0.97      0.98      3723
           6       1.00  

In [31]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(neigh, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8639034388521977
Report : 
              precision    recall  f1-score   support

           0       0.08      0.08      0.08       546
           1       0.00      0.00      0.00       466
           2       0.40      0.39      0.39      3303
           3       0.63      0.68      0.65      8857
           4       0.67      0.73      0.70      4850
           5       0.99      0.97      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.68      0.64      0.66      2773
           8       0.48      0.14      0.21       303
           9       0.00      0.00      0.00        37

    accuracy                           0.86     65865
   macro avg       0.49      0.46      0.47     65865
weighted avg       0.86      0.86      0.86     65865



In [32]:
#SVM
from sklearn.svm import SVC

clf = SVC(gamma='auto',decision_function_shape='ovo')
clf.fit(X_train, y_train)

SVC(decision_function_shape='ovo', gamma='auto')

In [33]:
y_pred=clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7428, 5: 3601, 3: 2175, 4: 1603, 2: 956, 7: 704})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [34]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0   28   38   59    0    0    6    0    0]
 [   0    0    6   36   70    0    0    5    0    0]
 [   0    0  423  225   94    3    3   38    0    0]
 [   0    0  374 1566  248    1    3   83    0    0]
 [   0    0   50   98 1006    1    4   53    0    0]
 [   0    0    4   88   25 3593    0   13    0    0]
 [   0    0    0    0    0    0 7418    0    0    0]
 [   0    0   66  103   88    3    0  463    0    0]
 [   0    0    5   15   12    0    0   43    0    0]
 [   0    0    0    6    1    0    0    0    0    0]]
Accuracy Score : 0.8786664237566041
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.44      0.54      0.49       786
           3       0.72      0.69      0.70      2275
           4       0.63      0.83      0.71      1212
           5       1.00      0.97      0.98      3723
           6       1.00  

In [35]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8778865861990435
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.45      0.51      0.48      3303
           3       0.71      0.69      0.70      8857
           4       0.63      0.83      0.72      4850
           5       1.00      0.96      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.64      0.67      0.66      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.88     65865
   macro avg       0.44      0.47      0.45     65865
weighted avg       0.87      0.88      0.87     65865



In [36]:
#Logistic Regression
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0,solver='saga',multi_class='multinomial').fit(X_train, y_train)

In [37]:
y_pred = clf.predict(X_test)
print(Counter(y_pred))
print(Counter(y_test))

Counter({6: 7418, 5: 3934, 3: 2129, 4: 1392, 2: 895, 7: 692, 1: 7})
Counter({6: 7418, 5: 3723, 3: 2275, 4: 1212, 2: 786, 7: 723, 0: 131, 1: 117, 8: 75, 9: 7})


In [38]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Accuracy Score : 0.8620878119876116
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.14      0.01      0.02       117
           2       0.43      0.49      0.46       786
           3       0.70      0.66      0.68      2275
           4       0.62      0.71      0.66      1212
           5       0.92      0.97      0.94      3723
           6       1.00      1.00      1.00      7418
           7       0.62      0.59      0.61       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.86     16467
   macro avg       0.44      0.44      0.44     16467
weighted avg       0.85      0.86      0.85     16467



In [39]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8594245805814924
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.17      0.01      0.02       466
           2       0.43      0.45      0.44      3303
           3       0.69      0.66      0.67      8857
           4       0.62      0.70      0.66      4850
           5       0.91      0.97      0.94     15148
           6       1.00      1.00      1.00     29582
           7       0.60      0.60      0.60      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.86     65865
   macro avg       0.44      0.44      0.43     65865
weighted avg       0.84      0.86      0.85     65865



In [10]:
#Multi Layer Perceptron
from sklearn.neural_network import MLPClassifier
clf = MLPClassifier(alpha=1)
clf.fit(X_train, y_train)
clf

MLPClassifier(alpha=1)

In [13]:
from sklearn.metrics import confusion_matrix 
from sklearn.metrics import accuracy_score 
from sklearn.metrics import classification_report 
y_pred = clf.predict(X_test)
results = confusion_matrix(y_test, y_pred) 
print('Confusion Matrix :')
print(results) 
print('Accuracy Score :',accuracy_score(y_test, y_pred))
print('Report : ')
print(classification_report(y_test, y_pred))

Confusion Matrix :
[[   0    0    1   56   66    2    0    6    0    0]
 [   0    0    0   38   72    2    0    5    0    0]
 [   0    0    8  620  108    7    0   43    0    0]
 [   0    0    6 1882  278   12    0   97    0    0]
 [   0    0    0  135  996    7    0   74    0    0]
 [   0    0    0   82   28 3598    0   15    0    0]
 [   0    0    0    0    0    0 7418    0    0    0]
 [   0    0    1  170  104    2    0  446    0    0]
 [   0    0    0    8   20    0    0   47    0    0]
 [   0    0    0    5    2    0    0    0    0    0]]
Accuracy Score : 0.8713183943644865
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.50      0.01      0.02       786
           3       0.63      0.83      0.71      2275
           4       0.59      0.82      0.69      1212
           5       0.99      0.97      0.98      3723
           6       1.00  

In [14]:
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10)
predicted = cross_val_predict(clf, X_train, y_train, cv=skf)
print('Accuracy Score :',accuracy_score(y_train, predicted))
print('Report : ')
print(classification_report(y_train, predicted))

Accuracy Score : 0.8707659606771426
Report : 
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.45      0.32      0.37      3303
           3       0.67      0.73      0.70      8857
           4       0.60      0.81      0.69      4850
           5       0.99      0.96      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.58      0.64      0.61      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.87     65865
   macro avg       0.43      0.45      0.43     65865
weighted avg       0.86      0.87      0.86     65865



In [15]:
#RANDOM FOREST REGRESSOR
from sklearn.ensemble import RandomForestRegressor
forest = RandomForestRegressor()
_ = forest.fit(X_train, y_train)
arr = forest.predict(X_train).astype(int)
print(classification_report(y_train, arr))

              precision    recall  f1-score   support

           0       0.88      0.75      0.81       546
           1       0.57      0.83      0.68       466
           2       0.38      0.90      0.54      3303
           3       0.65      0.46      0.54      8857
           4       0.81      0.57      0.67      4850
           5       0.98      0.97      0.97     15148
           6       0.98      1.00      0.99     29582
           7       0.90      0.72      0.80      2773
           8       0.50      0.00      0.01       303
           9       0.00      0.00      0.00        37

    accuracy                           0.86     65865
   macro avg       0.67      0.62      0.60     65865
weighted avg       0.88      0.86      0.86     65865



In [16]:
arr = forest.predict(X_test).astype(int)
print(classification_report(y_test, arr))

              precision    recall  f1-score   support

           0       0.19      0.06      0.09       131
           1       0.16      0.33      0.22       117
           2       0.26      0.66      0.37       786
           3       0.49      0.33      0.40      2275
           4       0.71      0.51      0.59      1212
           5       0.96      0.97      0.96      3723
           6       0.99      1.00      1.00      7418
           7       0.96      0.71      0.81       723
           8       1.00      0.01      0.03        75
           9       0.00      0.00      0.00         7

    accuracy                           0.82     16467
   macro avg       0.57      0.46      0.45     16467
weighted avg       0.85      0.82      0.82     16467



In [17]:
#NaiveBayes
from sklearn.naive_bayes import GaussianNB
classifier = GaussianNB()
classifier.fit(X_train, y_train)
y_pred  =  classifier.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.08      0.96      0.15       546
           1       0.00      0.02      0.00       466
           2       0.40      0.01      0.03      3303
           3       0.84      0.47      0.60      8857
           4       0.64      0.14      0.24      4850
           5       0.98      0.57      0.72     15148
           6       1.00      1.00      1.00     29582
           7       0.08      0.04      0.05      2773
           8       0.06      0.99      0.11       303
           9       0.02      0.78      0.03        37

    accuracy                           0.67     65865
   macro avg       0.41      0.50      0.29     65865
weighted avg       0.86      0.67      0.72     65865



In [18]:
y_pred  =  classifier.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.08      0.93      0.15       131
           1       0.00      0.04      0.01       117
           2       0.38      0.02      0.04       786
           3       0.87      0.49      0.63      2275
           4       0.63      0.13      0.22      1212
           5       0.99      0.56      0.71      3723
           6       1.00      1.00      1.00      7418
           7       0.09      0.04      0.06       723
           8       0.05      0.99      0.10        75
           9       0.01      0.57      0.02         7

    accuracy                           0.67     16467
   macro avg       0.41      0.48      0.29     16467
weighted avg       0.86      0.67      0.72     16467



In [19]:
#DecisionTree Entropy
from sklearn.tree import DecisionTreeClassifier
clf_entropy = DecisionTreeClassifier(criterion = "entropy", random_state = 100,max_depth = 3, min_samples_leaf = 5)
  
clf_entropy.fit(X_train, y_train)
y_pred = clf_entropy.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       546
           1       0.00      0.00      0.00       466
           2       0.00      0.00      0.00      3303
           3       0.42      0.99      0.59      8857
           4       0.60      0.07      0.12      4850
           5       1.00      0.96      0.98     15148
           6       1.00      1.00      1.00     29582
           7       0.00      0.00      0.00      2773
           8       0.00      0.00      0.00       303
           9       0.00      0.00      0.00        37

    accuracy                           0.81     65865
   macro avg       0.30      0.30      0.27     65865
weighted avg       0.78      0.81      0.76     65865



In [20]:
y_pred = clf_entropy.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00       131
           1       0.00      0.00      0.00       117
           2       0.00      0.00      0.00       786
           3       0.42      0.99      0.59      2275
           4       0.59      0.05      0.09      1212
           5       1.00      0.96      0.98      3723
           6       1.00      1.00      1.00      7418
           7       0.00      0.00      0.00       723
           8       0.00      0.00      0.00        75
           9       0.00      0.00      0.00         7

    accuracy                           0.81     16467
   macro avg       0.30      0.30      0.27     16467
weighted avg       0.78      0.81      0.76     16467

